In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import gym
import os
import cityflow
import gym_cityflow
import random

In [2]:
class DQN_network(torch.nn.Module):
    def __init__(self, input_dim, action_dim, hidden_dims=[128, 256, 128]):
        super().__init__()
        self.fc1 = torch.nn.Linear(input_dim, hidden_dims[0])
        self.norm1 = torch.nn.LayerNorm(hidden_dims[0])
        self.fc2 = torch.nn.Linear(hidden_dims[0], hidden_dims[1])
        self.norm2 = torch.nn.LayerNorm(hidden_dims[1])
        self.fc3 = torch.nn.Linear(hidden_dims[1], hidden_dims[2])
        self.norm3 = torch.nn.LayerNorm(hidden_dims[2])
        self.out = torch.nn.Linear(hidden_dims[2], action_dim)

    def forward(self, x):
        x = torch.relu(self.norm1(self.fc1(x)))
        x = torch.relu(self.norm2(self.fc2(x)))
        x = torch.relu(self.norm3(self.fc3(x)))
        return self.out(x)

In [3]:
CONFIG_PATH = "Intersections_4/sample_config.json"
CHECKPOINT = "checkpoints/qnet_double_dqn_final.pth"
NUM_AGENTS = 16
OBS_DIM    = 72
ACTION_DIM = 9
STEPS      = 1000

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
q_net = DQN_network(OBS_DIM, ACTION_DIM).to(device)
checkpoint = torch.load(CHECKPOINT, map_location=device)
q_net.load_state_dict(checkpoint)
q_net.eval()

DQN_network(
  (fc1): Linear(in_features=72, out_features=128, bias=True)
  (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  (fc2): Linear(in_features=128, out_features=256, bias=True)
  (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  (fc3): Linear(in_features=256, out_features=128, bias=True)
  (norm3): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  (out): Linear(in_features=128, out_features=9, bias=True)
)

In [5]:
env = gym.make(
    id="cityflow-v0",
    configPath=CONFIG_PATH,
    episodeSteps=STEPS
)
raw_obs = env.reset()
intersection_ids = sorted(raw_obs.keys())

/user/gaganvad/.local/lib/python3.10/site-packages/gym/utils/passive_env_checker.py:174: UserWarning: WARN: Future gym versions will require that `Env.reset` can be passed a `seed` instead of using `Env.seed` for resetting the environment random number generator.
  logger.warn(
/user/gaganvad/.local/lib/python3.10/site-packages/gym/utils/passive_env_checker.py:187: UserWarning: WARN: Future gym versions will require that `Env.reset` can be passed `options` to allow the environment initialisation to be passed additional information.
  logger.warn(
/user/gaganvad/.local/lib/python3.10/site-packages/gym/utils/passive_env_checker.py:195: UserWarning: WARN: The result returned by `env.reset()` was not a tuple of the form `(obs, info)`, where `obs` is a observation and `info` is a dictionary containing additional information. Actual type: `<class 'dict'>`
  logger.warn(


In [6]:
obs = [
    np.array(raw_obs[i], dtype=np.float32).flatten()
    for i in intersection_ids
]

In [7]:
total_reward     = 0.0
step_rewards     = []
queue_lengths    = []

prev_vehicles   = set(env.eng.get_vehicles(include_waiting=True))
arrived_vehicles = set()

lane_wait = env.eng.get_lane_waiting_vehicle_count()
queue_lengths.append(sum(lane_wait.values()))

for t in range(1, STEPS+1):
    actions = []
    for o in obs:
        tensor_o = torch.tensor(o, dtype=torch.float32, device=device).unsqueeze(0)
        with torch.no_grad():
            q_vals = q_net(tensor_o)
        actions.append(int(q_vals.argmax(dim=1).item()))
    
    next_raw_obs, rewards, done, info = env.step(actions)
    
    total_r = sum(r for _, r in rewards)
    total_reward += total_r
    step_rewards.append(total_r)

    cur_vehicles = set(env.eng.get_vehicles(include_waiting=True))
    new_arrivals = prev_vehicles - cur_vehicles
    arrived_vehicles |= new_arrivals
    prev_vehicles = cur_vehicles
    
    lw = env.eng.get_lane_waiting_vehicle_count()
    queue_lengths.append(sum(lw.values()))
    
    obs = [
        np.array(next_raw_obs[i], dtype=np.float32).flatten()
        for i in intersection_ids
    ]

    print(f"Step {t:4d}  reward {total_r:8.1f}  queue {queue_lengths[-1]:4d}")

    if done:
        print(f"Environment signaled done at step {t+1}")
        break

num_arrived     = len(arrived_vehicles)
total_wait_time = sum(queue_lengths)  
avg_wait_per_veh= total_wait_time/num_arrived if num_arrived>0 else float("nan")

avg_travel_time = env.eng.get_average_travel_time()  
active_vehicles = env.eng.get_vehicle_count()        

print("\n=== Evaluation Summary ===")
print(f"Total steps executed       : {t}")
print(f"Total reward accumulated   : {total_reward:.1f}")
print(f"Vehicles arrived           : {num_arrived}")
print(f"Active vehicles remaining  : {active_vehicles}")
print(f"Avg waiting time/vehicle   : {avg_wait_per_veh:.2f} s")
print(f"Avg travel time (all)      : {avg_travel_time:.2f} s")


/user/gaganvad/.local/lib/python3.10/site-packages/gym/utils/passive_env_checker.py:219: DeprecationWarning: WARN: Core environment is written in old step API which returns one bool instead of two. It is recommended to rewrite the environment with new step API. 
  logger.deprecation(
/user/gaganvad/.local/lib/python3.10/site-packages/gym/utils/passive_env_checker.py:146: UserWarning: WARN: The obs returned by the `step()` method was expecting a numpy array, actual type: <class 'list'>
  logger.warn(f"{pre} was expecting a numpy array, actual type: {type(obs)}")
/user/gaganvad/.local/lib/python3.10/site-packages/gym/utils/passive_env_checker.py:252: UserWarning: WARN: The reward returned by `step()` must be a float, int, np.integer or np.floating, actual type: <class 'list'>
  logger.warn(


Step    1  reward      0.0  queue    0
Step    2  reward      0.0  queue    0
Step    3  reward      0.0  queue    0
Step    4  reward      0.0  queue    0
Step    5  reward      0.0  queue    0
Step    6  reward      0.0  queue    0
Step    7  reward      0.0  queue    0
Step    8  reward      0.0  queue    0
Step    9  reward      0.0  queue    0
Step   10  reward      0.0  queue    0
Step   11  reward      0.0  queue    0
Step   12  reward      0.0  queue    0
Step   13  reward      0.0  queue    0
Step   14  reward      0.0  queue    0
Step   15  reward      0.0  queue    0
Step   16  reward      0.0  queue    0
Step   17  reward      0.0  queue    0
Step   18  reward      0.0  queue    0
Step   19  reward      0.0  queue    0
Step   20  reward      0.0  queue    0
Step   21  reward      0.0  queue    0
Step   22  reward      0.0  queue    0
Step   23  reward      0.0  queue    0
Step   24  reward      0.0  queue    0
Step   25  reward      0.0  queue    0
Step   26  reward      0.